# Notes

# Setup

In [67]:
print("hello")

hello


In [72]:
%load_ext autoreload
%autoreload 2

import os

import pandas as pd
import geopandas as gpd
from matplotlib import pyplot as plt
import warnings

import config
import hatfield_funcs as hf
from dask.distributed import Client
from urllib.parse import urlparse
from os import PathLike
import rasterio
from rasterio.errors import RasterioIOError
from rasterio.mask import mask
from pyproj import CRS
from contextlib import nullcontext
import numpy as np

from pathlib import Path
import sys

from my_extra_to_hatfield_funcs.io_funcs import (
    write_df_to_csv,
)

project_root = Path("../..").resolve()
sys.path.insert(0, str(project_root / "src"))

from biofilm_data.io import (
    load_biofilm_mean_resampled_spectra,
    load_imagery_ndvi_station_stats,
)

plt.style.use("default")
%config InlineBackend.figure_format = 'retina'


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [69]:
if os.getenv("GA_STORAGE_ACCOUNT_NAME") is None:
    raise RuntimeError(
        "This notebook is configured for Azure cloud storage. "
        "Set GA_STORAGE_ACCOUNT_NAME before running it."
    )

hf.config.configure_azure()
hf.config.configure_data_root_path(config.CONTAINER_NAME)

In [70]:
DASK_SCHEDULER_ADDRESS = os.getenv(
    "DASK_SCHEDULER_ADDRESS",
    "tcp://dask-wesa-fup-jchampion-scheduler:8786",
)
client = Client(DASK_SCHEDULER_ADDRESS)
print(client)

# Workers keep old copies of image_processor in memory; restart to drop them.
client.restart()
print("Workers:", len(client.scheduler_info()["workers"]))

# If config.SAS_URL exists, use it, otherwise ask for it
if hasattr(config, "SAS_URL"):
    SAS_URL = config.SAS_URL
else:
    SAS_URL = input("Enter SAS URL: ")
parsed_sas = urlparse(SAS_URL)
AZURE_STORAGE_ACCOUNT = parsed_sas.netloc.split(".")[0]
SAS_CONTAINER_NAME = parsed_sas.path.lstrip("/")
SAS_TOKEN = parsed_sas.query

if not SAS_CONTAINER_NAME or not SAS_TOKEN:
    raise ValueError("SAS_URL is not a valid container SAS URL.")


def bootstrap_worker_storage():
    """Configure GDAL Azure access inside each Dask worker."""
    import os

    os.environ["AZURE_STORAGE_ACCOUNT"] = AZURE_STORAGE_ACCOUNT
    os.environ["AZURE_STORAGE_SAS_TOKEN"] = SAS_TOKEN
    os.environ["CPL_VSIL_USE_TEMP_FILE_FOR_RANDOM_WRITE"] = "YES"
    os.environ["IGNORE_COG_LAYOUT_BREAK"] = "YES"
    os.environ["GDAL_CACHEMAX"] = "512"


bootstrap_worker_storage()
client.run(bootstrap_worker_storage)
client.register_worker_callbacks(setup=bootstrap_worker_storage)

# Client, scheduler, and workers should use compatible Dask/NumPy versions.
versions = client.get_versions(check=False)
print("Dask version information collected.")


<Client: 'tcp://100.112.11.145:8786' processes=5 threads=30, memory=60.00 GiB>


/home/coder/jchampion/Biofilm_LEG13553/confidential-leg13553-biofilm/.venv/lib/python3.10/site-packages/distributed/client.py:1595: VersionMismatchWarning: Mismatched versions found

+-------------+----------+-----------+----------+
| Package     | Client   | Scheduler | Workers  |
+-------------+----------+-----------+----------+
| dask        | 2026.8.0 | 2025.9.0  | 2025.9.0 |
| distributed | 2026.8.0 | 2025.9.0  | 2025.9.0 |
| lz4         | None     | 4.4.5     | 4.4.5    |
| msgpack     | 1.2.2    | 1.2.1     | 1.2.1    |
| numpy       | 2.2.6    | 2.1.3     | 2.1.3    |
| tornado     | 6.5.10   | 6.5.8     | 6.5.8    |
+-------------+----------+-----------+----------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Workers: 5
Dask version information collected.


# Parameters

In [6]:
force_redo = False

# Functions to Externalize

In [7]:
# Container path


def build_vsiaz_path(
    blob_path: str | Path,
    *,
    container: str = config.CONTAINER_NAME,
) -> str:
    """
    Build a GDAL ``/vsiaz/`` path for a blob in Azure Storage.

    Backslashes are converted to forward slashes, and leading or trailing
    slashes are removed from the container and blob path.

    Parameters
    ----------
    blob_path : str or pathlib.Path
        Path to the blob within the Azure Storage container.

    container : str, default CONTAINER_NAME
        Name of the Azure Storage container.

    Returns
    -------
    str
        Path in the form ``/vsiaz/<container>/<blob_path>``.

    Raises
    ------
    ValueError
        If ``container`` or ``blob_path`` is empty.

    Examples
    --------
    >>> build_vsiaz_path("rasters/image.tif", container="imagery")
    '/vsiaz/imagery/rasters/image.tif'

    >>> build_vsiaz_path(Path("rasters/image.tif"), container="imagery")
    '/vsiaz/imagery/rasters/image.tif'
    """
    container = container.strip("/")
    normalized_path = str(blob_path).replace("\\", "/").strip("/")

    if not container:
        raise ValueError("container must not be empty.")

    if not normalized_path:
        raise ValueError("blob_path must not be empty.")

    return f"/vsiaz/{container}/{normalized_path}"


def vsiaz_raster_is_accessible(
    path: str | PathLike[str],
) -> bool:
    """
    Return whether Rasterio can currently open a ``/vsiaz/`` raster.

    GDAL caching is disabled for the specified path so that files created
    during the current process can be detected.

    Parameters
    ----------
    path : str or path-like
        GDAL ``/vsiaz/`` path to check.

    Returns
    -------
    bool
        True if Rasterio can open the raster; otherwise False.
    """
    path = str(path)

    with rasterio.Env(CPL_VSIL_CURL_NON_CACHED=path):
        try:
            with rasterio.open(path, sharing=False):
                return True
        except RasterioIOError:
            return False


# Functions to Keep Here?

In [8]:
import tempfile
import uuid
from pathlib import Path
from typing import Any, Callable, Optional, Sequence, Union

import rasterio
from rasterio.windows import Window
from rasterio.windows import transform as window_transform


# ---------------------------------------------------------------------------
# Worker-side functions: self-contained (imports inside), shipped by value.
# ---------------------------------------------------------------------------


def _compute_tile(
    input_specs,
    window_spec,
    tile_spec,
    calculation,
    calculation_kwargs,
    pass_transform,
    out_nodata,
    out_dtype,
):
    """Read the bands of every input for one tile, apply `calculation`, return (1, h, w) `out_dtype`."""
    import numpy as np
    import rasterio
    from rasterio.windows import Window

    tile_row, tile_col, tile_height, tile_width = tile_spec
    window = Window(
        window_spec["col0"] + tile_col,
        window_spec["row0"] + tile_row,
        tile_width,
        tile_height,
    )

    arrays, tile_transform = [], None
    for path, bands in input_specs:
        with rasterio.open(path) as src:
            arrays.append(
                src.read(list(bands), window=window, masked=True, out_dtype="float32")
            )
            if tile_transform is None:
                tile_transform = src.window_transform(window)
    data = np.ma.concatenate(arrays, axis=0) if len(arrays) > 1 else arrays[0]

    kwargs = dict(calculation_kwargs)
    if pass_transform:
        kwargs["transform"] = tile_transform
    result = calculation(data, **kwargs)
    result = np.ma.filled(np.ma.asarray(result, dtype="float32"), np.nan)
    result = np.asarray(result, dtype="float32")

    if result.ndim == 2:
        result = result[np.newaxis]
    if result.shape != (1, tile_height, tile_width):
        raise ValueError(
            f"calculation must return (h, w) or (1, h, w) = "
            f"{(tile_height, tile_width)}; got {result.shape}."
        )

    # Any masked input band, or a non-finite result, becomes nodata.
    invalid = np.ma.getmaskarray(data).any(axis=0)[np.newaxis] | ~np.isfinite(result)

    dtype = np.dtype(out_dtype)
    if dtype.kind in "ui":
        info = np.iinfo(dtype)
        values = result[~invalid]
        if values.size and (values.min() < info.min or values.max() > info.max):
            raise ValueError(f"calculation results do not fit in {dtype.name}.")
        result = np.rint(result)

    return np.where(invalid, out_nodata, result).astype(dtype)


def _create_gtiff(path, profile, band_description):
    import rasterio

    with rasterio.open(path, "w", **profile) as dst:
        dst.set_band_description(1, band_description)
    return path


def _write_tile(previous_write, path, tile_spec, data):
    """Write one tile; `previous_write` only chains tasks so writes are serialized."""
    import rasterio
    from rasterio.windows import Window

    del previous_write
    tile_row, tile_col, tile_height, tile_width = tile_spec
    window = Window(tile_col, tile_row, tile_width, tile_height)

    with rasterio.open(path, "r+") as dst:
        dst.write(data, window=window)
    return tile_spec


def _finalize_to_cog(tmp_path, output_path, compress, blocksize, overviews):
    """Compute stats of the temporary GeoTIFF, then copy it to a COG."""
    import numpy as np
    import rasterio
    from rasterio.shutil import copy as rio_copy

    count, total, v_sum, v_min, v_max = 0, 0, 0.0, np.inf, -np.inf
    with rasterio.open(tmp_path) as src:
        for _, window in src.block_windows(1):
            block = src.read(1, window=window)
            total += block.size
            valid = block[np.isfinite(block) & (block != src.nodata)]
            if valid.size:
                count += valid.size
                v_sum += float(valid.sum(dtype=np.float64))
                v_min = min(v_min, float(valid.min()))
                v_max = max(v_max, float(valid.max()))

    if count == 0:
        raise RuntimeError("No valid pixels in output; check inputs and calculation.")

    rio_copy(
        tmp_path,
        output_path,
        driver="COG",
        compress=compress,
        blocksize=blocksize,
        OVERVIEWS=overviews,
        BIGTIFF="IF_SAFER",
        NUM_THREADS="1",
    )
    return {
        "valid_pixels": count,
        "min": v_min,
        "max": v_max,
        "mean": v_sum / count,
        "fraction_nodata": 1 - count / total,
    }


def _remove_file(path):
    import os

    try:
        os.remove(path)
    except FileNotFoundError:
        pass
    return True


# ---------------------------------------------------------------------------
# Example calculations: (masked array (n_bands, h, w), **kwargs) -> (h, w).
# NaN in the result is written as nodata.
# ---------------------------------------------------------------------------


def ndvi_calculation(
    data, *, red_index_one_based=1, nir_index_one_based=2, zero_tolerance=1e-8
):
    """NDVI; red_index and nir_index are positions along axis 0 of `data`."""
    import numpy as np

    red = np.ma.getdata(data[red_index_one_based - 1]).astype("float32")
    nir = np.ma.getdata(data[nir_index_one_based - 1]).astype("float32")
    numerator = nir - red
    denominator = nir + red

    out = np.full(red.shape, np.nan, dtype="float32")
    ok = np.abs(denominator) > zero_tolerance
    out[ok] = numerator[ok] / denominator[ok]
    return out


def spectral_angle_calculation(data, *, reference, degrees=True):
    """Spectral angle between each pixel and `reference` (one value per band read)."""
    import numpy as np

    values = np.ma.getdata(data).astype("float32")
    reference = np.asarray(reference, dtype="float32")
    if reference.shape != (values.shape[0],):
        raise ValueError(
            f"reference has shape {reference.shape}; expected ({values.shape[0]},)."
        )

    dot = np.einsum("bhw,b->hw", values, reference)
    norm = np.sqrt(np.einsum("bhw,bhw->hw", values, values))
    denominator = norm * np.linalg.norm(reference)

    out = np.full(dot.shape, np.nan, dtype="float32")
    ok = denominator > 0
    angle = np.arccos(np.clip(dot[ok] / denominator[ok], -1, 1))
    out[ok] = np.degrees(angle) if degrees else angle
    return out


def extent_mask(geometries, shape, transform, all_touched=False):
    """True inside any of the GeoJSON-like `geometries` (raster CRS); reusable helper."""
    from rasterio.features import geometry_mask

    return geometry_mask(
        geometries,
        out_shape=shape,
        transform=transform,
        invert=True,
        all_touched=all_touched,
    )


def biofilm_classification_calculation(
    data,
    *,
    ndvi_lower,
    ndvi_upper,
    sa_threshold,
    ndvi_index_one_based=1,
    sa_index_one_based=2,
    extent_geometries=None,
    all_touched=False,
    transform=None,
):
    """
    Class image: 1 * (NDVI < lower) + 2 * (NDVI > upper) + 4 * (SA > threshold).

    0 = possible biofilm. Pixels outside `extent_geometries` (if given) are nodata.
    Needs `pass_transform=True` when `extent_geometries` is given.
    """
    import numpy as np

    ndvi = np.ma.getdata(data[ndvi_index_one_based - 1])
    sa = np.ma.getdata(data[sa_index_one_based - 1])

    out = (
        (ndvi < ndvi_lower) * 1.0
        + (ndvi > ndvi_upper) * 2.0
        + (sa > sa_threshold) * 4.0
    ).astype("float32")

    if extent_geometries is not None:
        out[~extent_mask(extent_geometries, out.shape, transform, all_touched)] = np.nan
    return out


# ---------------------------------------------------------------------------
# Client-side driver
# ---------------------------------------------------------------------------


def process_imagery_in_tiles_with_callable(
    client: Any,
    *,
    calculation: Callable,
    input_path: Union[str, Path, Sequence[Union[str, Path]]],
    output_path: Union[str, Path],
    calculation_kwargs: Optional[dict] = None,
    pass_transform: bool = False,
    read_bands_one_based: Optional[Sequence] = None,
    exclude_bands_one_based: Sequence[int] = (),
    tile_size: int = 1024,
    row0: int = 0,
    col0: int = 0,
    height: Optional[int] = None,
    width: Optional[int] = None,
    compress: str = "LZW",
    blocksize: int = 256,
    overviews: str = "NONE",
    out_dtype: str = "float32",
    out_nodata: Optional[float] = None,
    band_description: str = "result",
    max_in_flight: int = 12,
    max_pending_writes: int = 16,
) -> dict:
    """
    Apply a user-defined function tile by tile on Dask workers and write a
    single-band COG.

    The function is shipped to workers by value, so it must import what it
    needs inside its body and must not rely on notebook globals.

    Parameters
    ----------
    client : dask.distributed.Client
    calculation : callable
        ``calculation(data, **calculation_kwargs)`` where ``data`` is a masked
        float32 array shaped (n_read_bands, tile_h, tile_w), bands in the order
        of ``read_bands_one_based`` (inputs stacked in the order given). Must
        return (tile_h, tile_w) or (1, tile_h, tile_w). Pixels where any read
        band is masked, or where the result is not finite, are written as
        ``out_nodata``.
    input_path : str, Path, or sequence of them
        Raster path(s) (``/vsiaz/...`` or local). Several inputs must share the
        same grid; their bands are stacked along axis 0 in the order given.
    output_path : str or Path
        Output COG path.
    calculation_kwargs : dict, optional
        Extra keyword arguments for ``calculation`` (must be picklable).
    pass_transform : bool, default False
        If True, ``calculation`` also receives ``transform=`` (affine transform
        of the tile, in the raster CRS), e.g. to rasterize a polygon.
    read_bands_one_based : sequence of int, optional
        One-based source bands to read, in the order passed to ``calculation``.
        Default: all bands except ``exclude_bands_one_based``. With several
        inputs, give one such sequence per input (or None for all bands).
    exclude_bands_one_based : sequence of int, default ()
        One-based source bands to skip when ``read_bands_one_based`` is None
        (single input only).
    out_dtype : str, default "float32"
        Output data type, e.g. "float32" or "uint8". Integer outputs are
        rounded and must fit in the type.
    out_nodata : float, optional
        Output nodata value. Default: -9999 for float types, the maximum
        value of the type (e.g. 255 for uint8) for integer types.
    tile_size, row0, col0, height, width, compress, blocksize, overviews,
    band_description, max_in_flight, max_pending_writes
        Same meaning as in ``image_processor``.

    Returns
    -------
    dict
        output_path, total_tiles, valid_pixels, min, max, mean, fraction_nodata.
    """
    import numpy as np
    from dask.distributed import as_completed, wait

    if not callable(calculation):
        raise TypeError("calculation must be callable.")
    if blocksize <= 0 or blocksize % 16 != 0:
        raise ValueError("blocksize must be positive and divisible by 16.")

    dtype = np.dtype(out_dtype)
    if dtype.kind not in "fui":
        raise ValueError(
            f"out_dtype must be a float or integer type; got {dtype.name}."
        )
    if out_nodata is None:
        out_nodata = -9999.0 if dtype.kind == "f" else float(np.iinfo(dtype).max)
    if dtype.kind in "ui":
        info = np.iinfo(dtype)
        if not (float(out_nodata).is_integer() and info.min <= out_nodata <= info.max):
            raise ValueError(f"out_nodata={out_nodata} is not valid for {dtype.name}.")

    input_paths = (
        [str(input_path)]
        if isinstance(input_path, (str, Path))
        else [str(p) for p in input_path]
    )
    output_path = str(output_path)
    calculation_kwargs = dict(calculation_kwargs or {})
    multiple = len(input_paths) > 1

    if multiple and exclude_bands_one_based:
        raise ValueError(
            "exclude_bands_one_based is only supported for a single input."
        )
    if (
        multiple
        and read_bands_one_based is not None
        and len(read_bands_one_based) != len(input_paths)
    ):
        raise ValueError("read_bands_one_based needs one entry per input raster.")

    input_specs, grid = [], None
    for i, path in enumerate(input_paths):
        with rasterio.open(path) as src:
            this_grid = (src.width, src.height, src.transform, src.crs)
            if grid is None:
                grid = this_grid
                source_profile = src.profile.copy()
            elif this_grid != grid:
                raise ValueError(f"{path} is not on the same grid as {input_paths[0]}.")

            requested = (
                (read_bands_one_based[i] if multiple else read_bands_one_based)
                if read_bands_one_based is not None
                else None
            )
            if requested is None:
                bands = tuple(
                    b
                    for b in range(1, src.count + 1)
                    if b not in set(exclude_bands_one_based)
                )
            else:
                bands = tuple(int(b) for b in requested)
            if not bands or any(b < 1 or b > src.count for b in bands):
                raise ValueError(
                    f"Bands {bands} invalid for {path} ({src.count} bands)."
                )
            input_specs.append((path, bands))

    src_width, src_height, src_transform, _ = grid
    win_height = src_height - row0 if height is None else int(height)
    win_width = src_width - col0 if width is None else int(width)
    if (
        row0 < 0
        or col0 < 0
        or win_height <= 0
        or win_width <= 0
        or row0 + win_height > src_height
        or col0 + win_width > src_width
    ):
        raise ValueError("Processing window is empty or outside the raster.")

    output_profile = source_profile
    output_profile.update(
        driver="GTiff",
        width=win_width,
        height=win_height,
        count=1,
        dtype=dtype.name,
        nodata=float(out_nodata),
        transform=window_transform(
            Window(col0, row0, win_width, win_height), src_transform
        ),
        tiled=True,
        blockxsize=blocksize,
        blockysize=blocksize,
        compress=compress,
        interleave="band",
        BIGTIFF="IF_SAFER",
    )
    output_profile.pop("photometric", None)

    window_spec = {"row0": row0, "col0": col0, "height": win_height, "width": win_width}
    tile_specs = [
        (r, c, min(tile_size, win_height - r), min(tile_size, win_width - c))
        for r in range(0, win_height, tile_size)
        for c in range(0, win_width, tile_size)
    ]
    total_tiles = len(tile_specs)

    # scheduler_info() can lack "workers" after idle/reconnect; nthreads() queries the scheduler.
    client.wait_for_workers(1, timeout=60)
    workers = list(client.nthreads())
    if not workers:
        raise RuntimeError("No Dask workers available.")
    writer = workers[0]  # Intermediate file lives on this worker's /tmp.

    print(
        f"Region: {win_height} x {win_width}, inputs/bands: {input_specs}, "
        f"output: {dtype.name}, tiles: {total_tiles}"
    )

    tmp_path = str(
        Path(tempfile.gettempdir()) / f"processing_{uuid.uuid4().hex[:10]}.tif"
    )
    client.submit(
        _create_gtiff,
        tmp_path,
        output_profile,
        band_description,
        workers=[writer],
        pure=False,
    ).result()

    tile_iter = iter(tile_specs)
    in_flight = {}

    def submit_compute(tile_spec):
        future = client.submit(
            _compute_tile,
            input_specs,
            window_spec,
            tile_spec,
            calculation,
            calculation_kwargs,
            pass_transform,
            out_nodata,
            dtype.name,
            pure=False,
        )
        in_flight[future] = tile_spec
        return future

    for _ in range(min(max_in_flight, total_tiles)):
        submit_compute(next(tile_iter))

    last_write, pending_writes, done_tiles = None, [], 0
    try:
        completed = as_completed(list(in_flight))
        for compute_future in completed:
            tile_spec = in_flight.pop(compute_future)
            error = compute_future.exception()
            if error is not None:
                raise RuntimeError(
                    f"Compute failed for tile {tile_spec}: "
                    f"{type(error).__name__}: {error}"
                ) from error

            last_write = client.submit(
                _write_tile,
                last_write,
                tmp_path,
                tile_spec,
                compute_future,
                workers=[writer],
                pure=False,
            )
            pending_writes.append(last_write)
            done_tiles += 1
            if done_tiles % max(1, total_tiles // 10) == 0:
                print(f"Computed {done_tiles}/{total_tiles} tiles", flush=True)

            # Backpressure: let half of the queued writes finish.
            if len(pending_writes) >= max_pending_writes:
                half = max_pending_writes // 2
                wait(pending_writes[:half])
                for f in pending_writes[:half]:
                    f.result()
                pending_writes = pending_writes[half:]

            try:
                completed.add(submit_compute(next(tile_iter)))
            except StopIteration:
                pass

        if pending_writes:
            wait(pending_writes)
            for f in pending_writes:
                f.result()

        print("Computing stats and writing COG...", flush=True)
        stats = client.submit(
            _finalize_to_cog,
            tmp_path,
            output_path,
            compress,
            blocksize,
            overviews,
            workers=[writer],
            pure=False,
        ).result()
    finally:
        try:
            client.submit(_remove_file, tmp_path, workers=[writer], pure=False).result()
        except Exception as cleanup_error:
            print(f"Warning: could not remove {tmp_path}: {cleanup_error}")

    return {"output_path": output_path, "total_tiles": total_tiles, **stats}


In [ ]:
def extract_horizontal_crs(crs: Any) -> CRS:
    """
    Return the horizontal component of a coordinate reference system.

    If the input is a compound CRS, its projected or geographic component
    is returned. Otherwise, the parsed CRS is returned unchanged.

    Parameters
    ----------
    crs : Any
        Any CRS representation accepted by ``pyproj.CRS.from_user_input``.

    Returns
    -------
    pyproj.CRS
        The horizontal CRS, or the original CRS if it is not compound.

    Raises
    ------
    ValueError
        If a compound CRS does not have exactly one projected or geographic
        component.
    """
    parsed_crs = CRS.from_user_input(crs)

    if not parsed_crs.is_compound:
        return parsed_crs

    horizontal_components = [
        component
        for component in parsed_crs.sub_crs_list
        if component.is_projected or component.is_geographic
    ]

    if len(horizontal_components) != 1:
        raise ValueError(
            "Expected exactly one horizontal component in compound CRS, "
            f"but found {len(horizontal_components)}: {parsed_crs}"
        )

    return horizontal_components[0]


def compact_crs_label(crs: Any) -> str:
    """
    Return a compact, human-readable label for a CRS.

    A recognized authority identifier, such as ``"EPSG:4326"``, is
    preferred. If no authority identifier is available, the CRS name
    is returned.

    Parameters
    ----------
    crs : Any
        Any CRS representation accepted by ``pyproj.CRS.from_user_input``.

    Returns
    -------
    str
        The CRS authority identifier or, if unavailable, its name.
    """
    parsed_crs = CRS.from_user_input(crs)
    authority = parsed_crs.to_authority()

    if authority is not None:
        authority_name, authority_code = authority
        return f"{authority_name}:{authority_code}"

    return parsed_crs.name


def extract_spectra_in_polygon(
    raster: str | PathLike[str] | Any,
    geometry: Any,
    *,
    band_indexes: tuple[int, ...],
    wavelengths_nm: tuple[int | float, ...],
    all_touched: bool = False,
) -> pd.DataFrame:
    """
    Extract valid pixel spectra from within a polygon.

    Each returned row represents one raster pixel, and each column represents
    one selected band labelled by its wavelength. A pixel is included only
    when all selected bands are unmasked and finite.

    Parameters
    ----------
    raster : path-like or open Rasterio dataset
        Raster to read. If an open dataset is supplied, it is not closed by
        this function.

    geometry : GeoJSON-like geometry or object with ``__geo_interface__``
        Polygon used to select pixels. The geometry must be in the same
        coordinate reference system as the raster.

    band_indexes : tuple of int
        One-based raster band indexes to read.

    wavelengths_nm : tuple of int or float
        Wavelength label for each selected band. Must have the same length as
        ``band_indexes``.

    all_touched : bool, default False
        If True, include every pixel touched by the polygon. If False, use
        Rasterio's default pixel-centre selection behaviour.

    Returns
    -------
    pandas.DataFrame
        Extracted spectra with one row per valid pixel and wavelengths as
        column labels. An empty DataFrame is returned when the selected area
        contains no pixels that are valid across all selected bands.

    Raises
    ------
    ValueError
        If no bands are requested, the number of bands and wavelengths
        differs, a band index is outside the raster, or the geometry does
        not overlap the raster.
    """
    if not band_indexes:
        raise ValueError("At least one band index must be provided.")

    if len(band_indexes) != len(wavelengths_nm):
        raise ValueError("band_indexes and wavelengths_nm must have equal length.")

    def empty_result() -> pd.DataFrame:
        result = pd.DataFrame(
            columns=wavelengths_nm,
            dtype="float64",
        )
        result.index.name = "pixel_id"
        return result

    raster_context = (
        rasterio.open(raster)
        if isinstance(raster, (str, PathLike))
        else nullcontext(raster)
    )

    with raster_context as src:
        invalid_indexes = [
            index for index in band_indexes if index < 1 or index > src.count
        ]

        if invalid_indexes:
            raise ValueError(
                f"Band indexes {invalid_indexes} are outside the valid "
                f"range 1–{src.count}."
            )

        data, _ = mask(
            src,
            [geometry],
            indexes=list(band_indexes),
            crop=True,
            filled=False,
            all_touched=all_touched,
        )

    masked_values = np.ma.getmaskarray(data)
    finite_values = np.isfinite(data.data)

    valid_pixels = ~masked_values.any(axis=0) & finite_values.all(axis=0)

    if not valid_pixels.any():
        return empty_result()

    spectra = data.data[:, valid_pixels].T.astype(
        "float64",
        copy=False,
    )

    return pd.DataFrame(
        spectra,
        columns=wavelengths_nm,
    ).rename_axis("pixel_id")


def analysis_band_indices(raster_band_count: int) -> tuple[int, ...]:
    """
    Return the 1-based raster band indices used in the analysis.

    All bands are used for a 10-band raster. For an 11-band raster,
    band 5 is excluded.

    Parameters
    ----------
    raster_band_count : int
        Total number of bands in the raster. Must be 10 or 11.

    Returns
    -------
    tuple of int
        One-based band indices to include in the analysis.

    Raises
    ------
    ValueError
        If the raster does not contain 10 or 11 bands.
    """
    if raster_band_count == 10:
        return tuple(range(1, 11))

    if raster_band_count == 11:
        return (1, 2, 3, 4, 6, 7, 8, 9, 10, 11)

    raise ValueError(
        f"Expected a 10- or 11-band raster, but found {raster_band_count} bands."
    )


def safe_normalized_difference(
    band_a: pd.Series,
    band_b: pd.Series,
    *,
    zero_tolerance: float = 1e-8,
) -> pd.Series:
    """
    Calculate a normalized difference while avoiding unstable division.

    The normalized difference is calculated as:

    ``(band_a - band_b) / (band_a + band_b)``

    Results are ``NaN`` where either input is non-finite or where the
    denominator is within ``zero_tolerance`` of zero.

    Parameters
    ----------
    band_a, band_b : pandas.Series
        Numeric bands with matching indices.

    zero_tolerance : float, default 1e-8
        Absolute tolerance used to identify denominators near zero.

    Returns
    -------
    pandas.Series
        Normalized difference with the same index as the input Series.

    Raises
    ------
    ValueError
        If the input indices differ or ``zero_tolerance`` is invalid.
    """
    if not band_a.index.equals(band_b.index):
        raise ValueError("band_a and band_b must have matching indices.")

    if not np.isfinite(zero_tolerance) or zero_tolerance < 0:
        raise ValueError("zero_tolerance must be a finite, non-negative number.")

    band_a = band_a.astype("float64")
    band_b = band_b.astype("float64")

    numerator = band_a - band_b
    denominator = band_a + band_b

    valid = (
        np.isfinite(band_a)
        & np.isfinite(band_b)
        & ~np.isclose(
            denominator,
            0.0,
            rtol=0.0,
            atol=zero_tolerance,
        )
    )

    return numerator.div(denominator.where(valid))

# Functions Here

In [10]:
def submit_job(
    client,
    input_path,
    output_path,
    callable_fun,
    read_bands_one_based=None,
    exclude_bands_one_based=None,
    calculation_kwargs=None,
    band_description="result",
):
    try:
        result = process_imagery_in_tiles_with_callable(
            client,
            calculation=callable_fun,
            input_path=input_path,
            output_path=output_path,
            read_bands_one_based=read_bands_one_based,
            exclude_bands_one_based=exclude_bands_one_based,
            calculation_kwargs=calculation_kwargs,
            band_description=band_description,
        )

        return {
            "status": "success",
            "valid_pixels": result["valid_pixels"],
            f"min": result["min"],
            f"max": result["max"],
            f"mean": result["mean"],
        }
        # print(f"  ✓ Success! {band_description} range: {result['min']:.3f} to {result['max']:.3f}")

    except Exception as e:
        return {
            "status": "failed",
            "error": str(e),
        }
        # print(f"  ✗ Failed: {e}")


def submit_jobs(
    client,
    force_redo,
    df_processing,
    callable_fun,
    read_bands_one_based=None,
    exclude_bands_one_based=None,
    calculation_kwargs=None,
    band_description="result",
):
    if force_redo:
        df_processing["output_ok"] = False

    results = {}
    df_jobs = df_processing.loc[
        ~df_processing["output_ok"].astype("boolean").fillna(False)
    ]

    print(f"Submitting {len(df_jobs)} jobs for computation of {band_description} ...")

    for (site, round_, dls_mode), job in df_jobs.iterrows():
        job_key = f"{site}_{round_}_{dls_mode}"
        print(f"\t{job_key} ...")
        try:
            results[job_key] = submit_job(
                client,
                input_path=build_vsiaz_path(job["input_path"]),
                output_path=build_vsiaz_path(job["output_path"]),
                callable_fun=callable_fun,
                read_bands_one_based=read_bands_one_based,
                exclude_bands_one_based=exclude_bands_one_based,
                calculation_kwargs=calculation_kwargs,
                band_description=band_description,
            )
            print(
                f"\t\tSuccess for job {job_key} with output range {results[job_key]['min']:.3f} to {results[job_key]['max']:.3f}"
            )
        except Exception as e:
            print(f"\t\tFailed: {e}")

    df_results = pd.DataFrame(results).T
    return df_results


# Compute NDVI Images

In [11]:
df_ndvi_processing = pd.DataFrame(
    index=pd.MultiIndex.from_tuples(
        config.IMAGERY["df_11b_image_container_paths"].index, names=["site", "round", "dls_mode"]
    ),
)

for (site, round_, dls_mode), row in config.IMAGERY["df_11b_image_container_paths"].iterrows():
    # Output path (relative to container root)
    input_path = row["path"]
    output_path = f"{config.CONTAINER_DIR_PATHS['processed_imagery']}/ndvi_{site}_round_{round_}_{row['date'].strftime('%Y%m%d')}_{dls_mode}.tif"
    df_ndvi_processing.loc[(site, round_, dls_mode), "input_path"] = input_path
    df_ndvi_processing.loc[(site, round_, dls_mode), "output_path"] = output_path
    df_ndvi_processing.loc[(site, round_, dls_mode), "input_ok"] = (
        vsiaz_raster_is_accessible(build_vsiaz_path(input_path))
    )
    df_ndvi_processing.loc[(site, round_, dls_mode), "output_ok"] = (
        vsiaz_raster_is_accessible(build_vsiaz_path(output_path))
    )

display(df_ndvi_processing.head())

print(
    f"{len(df_ndvi_processing)} output images,\n\tincluding {df_ndvi_processing['output_ok'].sum()} already existing,\n\tand {len(df_ndvi_processing) - df_ndvi_processing['input_ok'].sum()} with input not accessible."
)

input_path  \
site      round dls_mode                                                      
brunswick 1     DLS       aerial_v3_COG/RTB2_10Band_Brunswick_20260504_D...   
                NoDLS     airborne_data_20260504_v2_COG/RTB2_10Band_Brun...   
          2     DLS       aerial_v3_COG/RTB2_10Band_Brunswick_20260518_D...   
                NoDLS     airborne_data_20260518_v2_COG/RTB2_10Band_Brun...   
westham   1     DLS       aerial_v3_COG/RTB2_10Band_Westham_20260504_DLS...   

                                                                output_path  \
site      round dls_mode                                                      
brunswick 1     DLS       data/processed/imagery/ndvi_brunswick_round_1_...   
                NoDLS     data/processed/imagery/ndvi_brunswick_round_1_...   
          2     DLS       data/processed/imagery/ndvi_brunswick_round_2_...   
                NoDLS     data/processed/imagery/ndvi_brunswick_round_2_...   
westham   1     DLS       data/processed/imagery/ndvi_westham_round_1_20...   

                         input_ok output_ok  
site      round dls_mode                     
brunswick 1     DLS          True      True  
                NoDLS        True      True  
          2     DLS          True      True  
                NoDLS        True      True  
westham   1     DLS          True      True

8 output images,
	including 8 already existing,
	and 0 with input not accessible.


In [12]:
df_results = submit_jobs(
    client=client,
    force_redo=force_redo,
    df_processing=df_ndvi_processing,
    callable_fun=ndvi_calculation,
    read_bands_one_based=[
        config.IMAGERY['red_band_index_1_when_11bands'],
        config.IMAGERY['nir_band_index_1_when_11bands'],
    ],
    calculation_kwargs={
        "red_index_one_based": 1,
        "nir_index_one_based": 2,
    },
    band_description="NDVI",
)

display(df_results)

Submitting 0 jobs for computation of NDVI ...


""


# Compute Spectral Angle Images

In [13]:
# Read reference spectrum over all cases
df_mean_biofilm_resampled_spectra = load_biofilm_mean_resampled_spectra(
    config.CONTAINER_FILE_PATHS["biofilm_mean_resampled_spectra_csv"]
)
mean_biofilm_resampled_spectrum = df_mean_biofilm_resampled_spectra.loc[('all', 'all')]

In [14]:
df_spectral_angle_processing = pd.DataFrame(
    index=pd.MultiIndex.from_tuples(
        config.IMAGERY["df_11b_image_container_paths"].index,
        names=["site", "round", "dls_mode"],
    ),
)

for (site, round_, dls_mode), row in config.IMAGERY[
    "df_11b_image_container_paths"
].iterrows():
    # Output path (relative to container root)
    input_path = row["path"]
    output_path = f"{config.CONTAINER_DIR_PATHS['processed_imagery']}/spectral_angle_{site}_round_{round_}_{row['date'].strftime('%Y%m%d')}_{dls_mode}.tif"
    df_spectral_angle_processing.loc[(site, round_, dls_mode), "input_path"] = (
        input_path
    )
    df_spectral_angle_processing.loc[(site, round_, dls_mode), "output_path"] = (
        output_path
    )
    df_spectral_angle_processing.loc[(site, round_, dls_mode), "input_ok"] = (
        vsiaz_raster_is_accessible(build_vsiaz_path(input_path))
    )
    df_spectral_angle_processing.loc[(site, round_, dls_mode), "output_ok"] = (
        vsiaz_raster_is_accessible(build_vsiaz_path(output_path))
    )

display(df_spectral_angle_processing.head())

print(
    f"{len(df_spectral_angle_processing)} output images,\n\tincluding {df_spectral_angle_processing['output_ok'].sum()} already existing,\n\tand {len(df_spectral_angle_processing) - df_spectral_angle_processing['input_ok'].sum()} with input not accessible."
)


input_path  \
site      round dls_mode                                                      
brunswick 1     DLS       aerial_v3_COG/RTB2_10Band_Brunswick_20260504_D...   
                NoDLS     airborne_data_20260504_v2_COG/RTB2_10Band_Brun...   
          2     DLS       aerial_v3_COG/RTB2_10Band_Brunswick_20260518_D...   
                NoDLS     airborne_data_20260518_v2_COG/RTB2_10Band_Brun...   
westham   1     DLS       aerial_v3_COG/RTB2_10Band_Westham_20260504_DLS...   

                                                                output_path  \
site      round dls_mode                                                      
brunswick 1     DLS       data/processed/imagery/spectral_angle_brunswic...   
                NoDLS     data/processed/imagery/spectral_angle_brunswic...   
          2     DLS       data/processed/imagery/spectral_angle_brunswic...   
                NoDLS     data/processed/imagery/spectral_angle_brunswic...   
westham   1     DLS       data/processed/imagery/spectral_angle_westham_...   

                         input_ok output_ok  
site      round dls_mode                     
brunswick 1     DLS          True      True  
                NoDLS        True      True  
          2     DLS          True      True  
                NoDLS        True      True  
westham   1     DLS          True      True

8 output images,
	including 8 already existing,
	and 0 with input not accessible.


In [15]:
df_results = submit_jobs(
    client=client,
    force_redo=force_redo,
    df_processing=df_spectral_angle_processing,
    callable_fun=spectral_angle_calculation,
    exclude_bands_one_based=[5],
    calculation_kwargs={
        "reference": mean_biofilm_resampled_spectrum,
        "degrees": True,
    },
    band_description="Spectral Angle (degrees)",
)

display(df_results)

Submitting 0 jobs for computation of Spectral Angle (degrees) ...


""


# Extract Station Spectra

## Read and Process Station Polygons (2x2m only)

In [34]:
gdf_station_polygons = gpd.read_file(
    build_vsiaz_path(config.CONTAINER_FILE_PATHS["gnss_station_polygons_geojson"])
)

gdf_station_polygons["station_id"] = (
    gdf_station_polygons["station_name"].astype("string").str.strip()
)

# Transform 'is_3x3m' to True or False
gdf_station_polygons["is_3x3m"] = gdf_station_polygons["is_3x3m"] == "Yes"

# Add site, mapping based on the sites in config.SITES and first letter in station_id
gdf_station_polygons["site"] = (
    gdf_station_polygons["station_id"]
    .str[0]
    .str.lower()
    .map({site[0].lower(): site for site in config.SITES})
)

gdf_station_polygons = gdf_station_polygons.query("`is_3x3m` == False")[["site", "round", "station_id", "geometry"]]
gdf_station_polygons = gdf_station_polygons.set_index(["site", "round", "station_id"]).sort_index()

gdf_station_polygons["sampled_after_flight"] = [
    station_id
    in config.IMAGERY["stations_sampled_after_flight_for_site_round"].get(
        (site, int(round_number)),
        [],
    )
    for site, round_number, station_id in gdf_station_polygons.index
]

display(gdf_station_polygons.head())

geometry  \
site      round station_id                                                      
brunswick 1     B1          POLYGON ((488833.82 5434224.912, 488835.82 543...   
                B10         POLYGON ((488875.848 5433865.401, 488877.848 5...   
                B11         POLYGON ((489056.793 5433627.776, 489058.793 5...   
                B12         POLYGON ((488897.659 5433305.857, 488899.659 5...   
                B13         POLYGON ((488555.946 5433166.387, 488557.946 5...   

                            sampled_after_flight  
site      round station_id                        
brunswick 1     B1                         False  
                B10                        False  
                B11                        False  
                B12                        False  
                B13                        False

## Extract Pixel Spectra

In [58]:
df_imagery_spectra = pd.DataFrame()

for index, row in config.IMAGERY["df_11b_image_container_paths"].iterrows():
    (site, round_, dls_mode) = index
    print(f"Extracting spectra for image {index}...")

    vsiaz_raster_path = build_vsiaz_path(row["path"])

    gdf_station_polygons_for_this_image = gdf_station_polygons.xs(
        (site, round_),
        level=("site", "round"),
    )

    frames = []

    with rasterio.open(vsiaz_raster_path) as src:
        if src.crs is None:
            raise ValueError(f"Raster has no CRS: {vsiaz_raster_path}")

        gdf_station_polygons_for_this_image_reprojected = gdf_station_polygons_for_this_image.to_crs(extract_horizontal_crs(src.crs))

        band_indexes = analysis_band_indices(src.count)

        for station_id, station in gdf_station_polygons_for_this_image_reprojected.iterrows():
            spectra = extract_spectra_in_polygon(
                src,
                station.geometry,
                wavelengths_nm=config.MICASENSE_10B_CENTER_WAVELENGTHS,
                band_indexes=band_indexes,
            )

            if spectra.empty:
                continue

            spectra = spectra.reset_index()

            spectra["site"] = site
            spectra["round"] = round_
            spectra["dls_mode"] = dls_mode
            spectra["station_id"] = station_id
            spectra["sampled_after_flight"] = station["sampled_after_flight"]

            frames.append(spectra)

    df_spectra_for_this_image = pd.concat(frames, ignore_index=True).set_index(
        ["site", "round", "dls_mode", "station_id", "sampled_after_flight", "pixel_id"]
    )
    df_imagery_spectra = pd.concat([df_imagery_spectra, df_spectra_for_this_image], ignore_index=False)


Extracting spectra for image ('brunswick', 1, 'DLS')...
Extracting spectra for image ('brunswick', 1, 'NoDLS')...
Extracting spectra for image ('brunswick', 2, 'DLS')...
Extracting spectra for image ('brunswick', 2, 'NoDLS')...
Extracting spectra for image ('westham', 1, 'DLS')...
Extracting spectra for image ('westham', 1, 'NoDLS')...
Extracting spectra for image ('westham', 2, 'DLS')...
Extracting spectra for image ('westham', 2, 'NoDLS')...


## Compute NDVI and Stats

In [65]:
df_imagery_ndvi = safe_normalized_difference(
    df_imagery_spectra[740],
    df_imagery_spectra[668],
)

df_imagery_ndvi_stats = (
    df_imagery_ndvi
    .groupby(["site", "round", "dls_mode", "station_id", "sampled_after_flight"])
    .agg(["mean", "std"])
    .rename(columns={"mean": "ndvi_mean", "std": "ndvi_std"})
)

display(df_imagery_ndvi_stats.head())

ndvi_mean  ndvi_std
site      round dls_mode station_id sampled_after_flight                     
brunswick 1     DLS      B1         False                  0.128304  0.011330
                         B10        False                  0.131698  0.020235
                         B11        False                  0.123708  0.030610
                         B12        False                  0.082977  0.014243
                         B13        False                  0.080841  0.016162

In [ ]:
# Save NDVI station stats to CSV
write_df_to_csv(
    df_imagery_ndvi_stats,
    config.CONTAINER_FILE_PATHS["imagery_ndvi_station_stats_csv"],
    index=True,
)

# Can be loaded back with
# load_imagery_ndvi_station_stats(config.CONTAINER_FILE_PATHS["imagery_ndvi_station_stats_csv"])

DataFrame saved to wesa-fup-restricted-027817p/data/processed/imagery/imagery_ndvi_station_stats.csv


# DEV

In [ ]:
# import math

# raster_path = build_vsiaz_path(config.IMAGERY["df_11b_image_container_paths"]["path"].iloc[0])

# with rasterio.open(raster_path) as src:
#     xres, yres = src.res

#     levels = [1, *src.overviews(1)]

#     for factor in levels:
#         print(
#             {
#                 "factor": factor,
#                 "resolution": (
#                     xres * factor,
#                     yres * factor,
#                 ),
#                 "size": (
#                     math.ceil(src.width / factor),
#                     math.ceil(src.height / factor),
#                 ),
#             }
#         )


{'factor': 1, 'resolution': (0.2999999999999989, 0.2999999999999869), 'size': (20812, 14196)}
{'factor': 2, 'resolution': (0.5999999999999978, 0.5999999999999738), 'size': (10406, 7098)}
{'factor': 4, 'resolution': (1.1999999999999955, 1.1999999999999476), 'size': (5203, 3549)}
{'factor': 8, 'resolution': (2.399999999999991, 2.399999999999895), 'size': (2602, 1775)}
{'factor': 16, 'resolution': (4.799999999999982, 4.79999999999979), 'size': (1301, 888)}
{'factor': 32, 'resolution': (9.599999999999964, 9.59999999999958), 'size': (651, 444)}


# END